# Conjunto de dados sobre doenças cardíacas

Este conjunto de dados data de 1988 e consiste em quatro bases de dados: Cleveland, Hungria, Suíça e Long Beach V. 

O conjunto de dados heart.csv contém 1.025 instâncias e 14 atributos:

+ idade (age)
+ sexo (sex)
+ tipo de dor no peito (cp)
+ pressão arterial em repouso (trestbps)
+ colesterol sérico (chol)
+ glicemia em jejum (fbs)
+ eletrocardiograma em repouso (restecg)
+ frequência cardíaca máxima atingida (thalach)
+ angina induzida por exercício (exang)
+ depressão do segmento ST (oldpeak)
+ inclinação do segmento ST (slope)
+ número de vasos principais coloridos por fluoroscopia (ca)
+ cintilografia (thal)
+ diagnóstico (target)


# Importar Bibliotecas

In [7]:
# biblioteca para ler os dados (ela também permite a manipulação, análise e tratamento de dados)
import pandas as pd
# (train_test_split) usada para dividir um conjunto de dados em subconjuntos de treinamento e teste.
from sklearn.model_selection import train_test_split
# padronizar as variáveis (features), transformando os dados de modo que tenham média igual a 0 e desvio padrão igual a 1 (utilizado para garantir que todas as variáveis tenham o mesmo peso inicial, acelerando a convergência e melhorando a precisão do algoritmo)
from sklearn.preprocessing import StandardScaler
# implementação da Rede Neural Artificial do tipo Perceptron Multicamadas 
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score


# Carregar os Dados

In [8]:
# Carregar o arquivo CSV por meio da biblioteca pandas
df = pd.read_csv('heart.csv')

# Visualizar as 05 primeiras linhas
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


# Seleção dos dados de entrada X e a saída y

In [9]:
# X contém todas as colunas exceto 'target' (selecionará do dataframe todas as colunas com excessão da coluna target)
X = df.drop(columns=['target'])

# y contém apenas a coluna alvo (seleciona apenas a coluna target)
y = df['target']

# Dividir os dados que serão utilizados no *treino* e *teste* do modelo

In [10]:
'''
X: As features (variáveis de entrada) que o modelo vai usar.
y: O target (variável de saída) que você quer prever.
X_train: 80% das características usadas para treinar o modelo.
X_test: 20% das características usadas para testar o modelo depois de treinado.
y_train: respostas correspondentes ao X_train.
y_test: espostas correspondentes ao X_test.

Na parametrização:

test_size=0.2: Separa 20% dos dados para teste e deixa os 80% restantes para o treino.
random_state=42: Fixar um número (como 42) garante que, toda vez que você rodar o código, a divisão dos dados seja exatamente igual.
stratify=y: Garante que a proporção das classes na variável y seja mantida de forma equilibrada tanto no treino quanto no teste

'''


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


print(f"Instâncias para treino: {X_train.shape[0]}")
print(f"Instâncias para teste: {X_test.shape[0]}")

Instâncias para treino: 820
Instâncias para teste: 205


# Padronização dos Dados 

As Redes Neurais utilizam funções de ativação e otimizadores baseados em gradiente que são muito sensíveis à escala das variáveis. Variáveis com magnitudes elevadas dominariam variáveis com intervalos menores

In [11]:
# Instancia o padronizador (cria uma instância (um objeto real na memória) a partir da classe StandardScaler)
scaler = StandardScaler()

# Calcula e ajusta as médias e o desvio padrão dos dados de treino e, ao mesmo tempo, padroniza esses dados para terem média zero e desvio padrão um
X_train_scaled = scaler.fit_transform(X_train)

# Aplica a mesma escala nos dados de teste
X_test_scaled = scaler.transform(X_test)

# Configuração da MLP

In [12]:
# Instancia o modelo com sua padronização
mlp = MLPClassifier(
    hidden_layer_sizes=(94, 75, 53, 74, 44),  # Duas camadas ocultas: a 1ª com 32 neurônios e a 2ª com 16
    activation='relu',            # Função de ativação não linear
    solver='adam',                # Otimizador estocástico baseado em gradiente
    max_iter=500,                 # Número máximo de épocas de treinamento
    random_state=42,
    early_stopping=True           # Interrompe se a perda de validação parar de melhorar
)

# Treinamento do Modelo

In [13]:
# Treinamento da rede
mlp.fit(X_train_scaled, y_train)

MLPClassifier(early_stopping=True, hidden_layer_sizes=(94, 75, 53, 74, 44),
              max_iter=500, random_state=42)

# Avaliação do Modelo

In [14]:
# Realizar predições no conjunto de teste
y_pred = mlp.predict(X_test_scaled)

# Acurácia global
acc = accuracy_score(y_test, y_pred)
print(f"Acurácia no Teste: {acc * 100:.2f}%\n")

Acurácia no Teste: 100.00%



# Testando novas previsões

In [18]:
import numpy as np

# Exemplo de dados de um novo paciente fictício (13 características)
novo_paciente = np.array([[55, 1, 1, 130, 240, 0, 1, 150, 0, 1.2, 1, 0, 2]])

# Escalar os dados com o mesmo scaler do treino
novo_paciente_scaled = scaler.transform(novo_paciente)

# Predição de classe (0 ou 1) e probabilidade
predicao = mlp.predict(novo_paciente_scaled)

# Calcular a probabilidade de um novo paciente pertencer a cada uma das classes possíveis
probabilidade = mlp.predict_proba(novo_paciente_scaled)

print(f"Diagnóstico Previsto: {'Positivo (1)' if predicao[0] == 1 else 'Negativo (0)'}")
print(f"Probabilidades [Classe 0, Classe 1]: {probabilidade[0]}")

Diagnóstico Previsto: Positivo (1)
Probabilidades [Classe 0, Classe 1]: [0.46910455 0.53089545]


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


# Atividade para próxima aula

1 - Ajustar o modelo para que tenha acurácia acima de **95%**;

2 - Utilizar a biblioteca **joblib** para salvar o modelo treinado. (Dica: lembre-se que os dados estão padronizados)

3 - Ajuste o exemplo:

```
import numpy as np

# Exemplo de dados de um novo paciente fictício (13 características)
novo_paciente = np.array([[55, 1, 1, 130, 240, 0, 1, 150, 0, 1.2, 1, 0, 2]])

# Escalar os dados com o mesmo scaler do treino
novo_paciente_scaled = scaler.transform(novo_paciente)

# Predição de classe (0 ou 1) e probabilidade
predicao = mlp.predict(novo_paciente_scaled)

# Calcular a probabilidade de um novo paciente pertencer a cada uma das classes possíveis
probabilidade = mlp.predict_proba(novo_paciente_scaled)

print(f"Diagnóstico Previsto: {'Positivo (1)' if predicao[0] == 1 else 'Negativo (0)'}")
print(f"Probabilidades [Classe 0, Classe 1]: {probabilidade[0]}")
```
Para ler o modelo salvo para realizar as previsões.

In [19]:
import joblib

# 1. Salvar o modelo treinado (contém pesos, bias e hiperparâmetros)
joblib.dump(mlp, 'modelo_mlp_heart.joblib')

# 2. Salvar o scaler ajustado
joblib.dump(scaler, 'scaler_heart.joblib')

print("Modelo e Scaler salvos com sucesso!")

Modelo e Scaler salvos com sucesso!


In [21]:
import numpy as np

# Carregar o modelo e o scaler
mlp_carregado = joblib.load('modelo_mlp_heart.joblib')
scaler_carregado = joblib.load('scaler_heart.joblib')

# Novo paciente de exemplo
novo_paciente = np.array([[55, 1, 1, 130, 240, 0, 1, 150, 0, 1.2, 1, 0, 2]])

# Escalar e prever
novo_paciente_scaled = scaler_carregado.transform(novo_paciente)
predicao = mlp_carregado.predict(novo_paciente_scaled)
probabilidade = mlp.predict_proba(novo_paciente_scaled)

print(f"Diagnóstico Previsto: {'Positivo (1)' if predicao[0] == 1 else 'Negativo (0)'}")
print(f"Probabilidades [Classe 0, Classe 1]: {probabilidade[0]}")

Diagnóstico Previsto: Positivo (1)
Probabilidades [Classe 0, Classe 1]: [0.46910455 0.53089545]


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
